# 自定义中间件-Node-style hooks

## 1、基本用法

### 1.1 基于装饰器的实现

In [2]:
from typing import Any
from langchain.agents.middleware import before_model, after_model, AgentState, before_agent, after_agent, \
    AgentMiddleware
from langgraph.runtime import Runtime
from langchain.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware, ContextEditingMiddleware, ClearToolUsesEdit, \
    FilesystemFileSearchMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from rich import print as rprint
from langgraph.checkpoint.memory import InMemorySaver

@before_model()
def before_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before_model <-----"
    return None

@after_model()
def after_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after_model <-----"
    return None

@before_agent()
def before_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before_agent <-----"
    return None

@after_agent()
def after_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after_agent <-----"
    return None

# 1.初始化模型
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    model="483025889/qwen3.5:9b", # ok
    # model="gemma4:12b", # 这里不好用
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

agent = create_agent(
    model=model,
    middleware=[before_model_middleware, after_model_middleware,before_agent_middleware, after_agent_middleware],
)

response = agent.invoke({
    "messages": [HumanMessage("2的12次方等于几")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

2的12次方等于几----> before_agent <---------> before_model <-----
================================== Ai Message ==================================

2 的 12 次方等于 **4096**。

计算过程简述：
$2^{12} = 2^{10} \times 2^2 = 1024 \times 4 = 4096$。----> after_model <---------> after_agent <-----


### 1.2基于类实现

In [1]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware,AgentState, before_agent, after_agent
from langgraph.runtime import Runtime
from langchain.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware, ContextEditingMiddleware, ClearToolUsesEdit, \
    FilesystemFileSearchMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from rich import print as rprint
from langgraph.checkpoint.memory import InMemorySaver

## 使用类的方式，方法名称是有讲究的，一定得是before_agent,after_agent,before_model,after_model
class MyMiddleware(AgentMiddleware): # 中间件类
    def before_model(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> before_model <-----"
        return None

    def after_model(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> after_model <-----"
        return None


    def before_agent(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> before_agent <-----"
        return None

    def after_agent(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> after_agent <-----"
        return None

middleware = MyMiddleware()

# 1.初始化模型
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    model="483025889/qwen3.5:9b", # ok
    # model="gemma4:12b", # 这里不好用
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

agent = create_agent(
    model=model,
    middleware=[middleware],
)

response = agent.invoke({
    "messages": [HumanMessage("2的12次方等于几")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

2的12次方等于几----> before_agent <---------> before_model <-----
================================== Ai Message ==================================

2的12次方等于 **4096**。  
计算过程：2^10 = 1024，因此 2^12 = 1024 × 2² = 1024 × 4 = **4096**。  

常见用途：计算机中常用此数值（如内存、存储容量单位过渡）。----> after_model <---------> after_agent <-----


## 2、装饰器参数：can_jump_to

### 2.1 基于装饰器的实现

In [1]:
from typing import Any
from langchain.agents.middleware import before_model, after_model, AgentState
from langgraph.runtime import Runtime
from langchain.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware, ContextEditingMiddleware, ClearToolUsesEdit, \
    FilesystemFileSearchMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from rich import print as rprint
from langgraph.checkpoint.memory import InMemorySaver

# 1.初始化模型
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    model="483025889/qwen3.5:9b", # ok
    # model="gemma4:12b", # 这里不好用
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

@tool
def get_news() -> str:
    """获取当日新闻"""
    return f"美加墨世界杯今日开幕"

# 在模型（LLM）执行前触发。允许跳转到 "tools" 节点。
@before_model(can_jump_to=["tools"])
def force_tool_first(state: AgentState,runtime:Runtime) -> dict[str, Any] | None:
    """
    【业务场景：强行拦截并触发工具】
    如果用户输入包含 "direct tool"，则跳过本次大模型的思考/生成阶段，
    直接伪造一个大模型的 tool_calls 意图，强行把控制权移交给工具执行节点。
    """
    text = state["messages"][-1].content
    if isinstance(text,str) and "direct tool" in text:
        print("[MIDDLEWARE] before_model: jump_to='tools'")

        # 人工构造一个大模型的消息对象（AIMessage）
        # 欺骗系统，让系统误以为这是模型自己决定要调用的工具
        fake_toolcall=AIMessage(
            content="人工构造消息",
            tool_calls=[
                {
                    "name":"get_news",
                    "args":{},
                     "id": "call_force_weather_001",
                }
            ]
        )

        # 返回更新后的状态：注入伪造的消息，并明确指定下一步跳转到 "tools" 节点
        return {
            "messages":[fake_toolcall],
            "jump_to":"tools",
        }
    # 如果不满足触发条件，返回 None，流程正常向下流转（继续让 LLM 思考）
    return None

# 在模型（LLM）执行生成之后触发。允许重新跳转回 "model" 节点。
@after_model(can_jump_to=["model"])
def retry_with_extra_instruction(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    【业务场景：反思/重试机制】
    如果大模型已经生成了回答，但发现用户最初的请求包含 "retry model"，
    则动态追加一条系统提示词（SystemMessage），强行让模型重新生成（重试）一次。
    """
    # 倒序遍历消息历史，找到最近的一次用户输入（human 消息）
    user_text = ""
    for msg in reversed(state["messages"]):
        if getattr(msg,"type","") == "human":
            user_text = getattr(msg,"content","")
            break
    # 检查用户输入是否包含触发重试的关键字
    if isinstance(user_text,str) and "retry model" in user_text.lower():
         # 【核心防御】：防止无限循环重跳（死循环）
         # 检查消息历史中是否已经注入过这条特殊的系统提示。如果有，说明已经重试过了，不再重复干预。
         already_injected = any(
           isinstance(getattr(msg,"content",None),str)
           and "你必须以【二次回答】开头" in msg.content
           for msg in state["messages"]
         )
         if already_injected:
             return None  # 已注入过，直接放行，结束重试流程
         print("[MIDDLEWARE] retry_with_extra_instruction: jump_to='model' with extra system instruction")
         # 返回更新后的状态：追加强力约束的系统消息，并将指针跳回 "model" 节点重新执行
         return {
             "messages":[SystemMessage("你必须以【二次回答】开头，并且只用一句话回答。")],
             "jump_to":"model",
         }


# 在模型（LLM）执行前触发。允许直接跳转到 "end" 节点（强行终止）。
@before_model(can_jump_to=["end"])
def overflow_context_processor(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    【业务场景：安全卫士/异常拦截】
    模拟上下文窗口溢出（Token超限）或其他严重的系统阻断情况。
    一旦触发，直接熔断流程，拒绝让大模型继续处理，直接报错或返回兜底文案。
    """
    # 假装溢出,模拟检查最后一条消息是否包含 overflow 标识
    if "overflow" in state["messages"][-1].content:
        print("[MIDDLEWARE] before_model: jump_to='end' when contenxt window overflow")

        # 构造兜底的结束消息，并直接指定跳转到 "end" 终止 Agent 运行
        return {
            "messages":[AIMessage("上下文窗口溢出，终止")],
            "jump_to":"end",
        }

agent = create_agent(
    model=model,
    tools=[get_news],
    # # 将定义的中间件按照顺序挂载到 Agent 中（注意：执行顺序会严格按照列表声明顺序）
    middleware=[force_tool_first,retry_with_extra_instruction,overflow_context_processor],
)

def run_once(user_input:str):
    resp = agent.invoke({
        "messages":[{"role": "user", "content": user_input}]
    })

    for msg in resp["messages"]:
        msg.pretty_print()

if __name__ == "__main__":
    # Case 1: 直接跳 tools
    # 预期表现：
    # 1. 触发 force_tool_first，打印 "[MIDDLEWARE] before_model: jump_to='tools'"
    # 2. 绕过 LLM 的首轮思考，直接调用 `get_news` 工具
    # 3. 工具返回结果后，LLM 总结工具结果并输出
    print('=' * 30, '-> Case 1 <-', '=' * 30)
    run_once("请帮我查今日新闻 direct tool")

    # Case 2: 输出后跳回 model
    # 预期表现：
    # 1. 正常进入 LLM 生成第 1 版回答
    # 2. 触发 retry_with_extra_instruction，打印 "[MIDDLEWARE] after_model: jump_to='model'..."
    # 3. 注入系统提示词后，LLM 被强行拉回并生成第 2 版回答
    # 4. 最终输出应带有“【二次回答】”前缀
    print('=' * 30, '-> Case 2 <-', '=' * 30)
    run_once("请随便介绍一下 LangChain retry model")

    # Case 3:
    # 预期表现：
    # 1. 触发 overflow_context_processor 中间件
    # 2. 直接打印终止信息并退出，LLM 根本不会接收到这个请求
    print('=' * 30, '-> Case 3 <-', '=' * 30)
    run_once("你好 overflow")
    # Case 4: 正常流程
    # 预期表现：
    # 1. 没有任何中间件被触发（不满足任何关键字）
    # 2. Agent 走正常的 OOTB（Out of the box）标准工作流：User -> Model -> Call Tool -> Model -> End
    print('=' * 30, '-> Case 4 <-', '=' * 30)
    run_once("今日新闻摘要？")

============================== -> Case 1 <- ==============================
[MIDDLEWARE] before_model: jump_to='tools'
================================ Human Message =================================

请帮我查今日新闻 direct tool
================================== Ai Message ==================================

人工构造消息
Tool Calls:
  get_news (call_force_weather_001)
 Call ID: call_force_weather_001
  Args:
================================= Tool Message =================================
Name: get_news

美加墨世界杯今日开幕
================================== Ai Message ==================================

以下是为您获取的今日热点新闻：

**新闻标题：** 美加墨世界杯今盛大开幕
**内容：** 今日正值国际足联世界杯开幕，备受瞩目的赛事在北美三国正式拉开帷幕。
============================== -> Case 2 <- ==============================
[MIDDLEWARE] retry_with_extra_instruction: jump_to='model' with extra system instruction
================================ Human Message =================================

请随便介绍一下 LangChain retry model
================================== Ai Message ========

### 2.2基于类实现

In [4]:
from typing import Any
from langchain.agents.middleware import before_model, after_model, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware, ContextEditingMiddleware, ClearToolUsesEdit, \
    FilesystemFileSearchMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from rich import print as rprint
from langgraph.checkpoint.memory import InMemorySaver

# 1.初始化模型
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    model="483025889/qwen3.5:9b", # ok
    # model="gemma4:12b", # 这里不好用
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

@tool
def get_news() -> str:
    """获取当日新闻"""
    return f"美加墨世界杯今日开幕"


class MyMiddleware(AgentMiddleware):
    # 在模型（LLM）执行前触发。允许跳转到 "tools" 节点。
    @hook_config(can_jump_to=["tools","end"])
    def force_tool_first(state: AgentState,runtime:Runtime) -> dict[str, Any] | None:
        """
        【业务场景：强行拦截并触发工具】
        如果用户输入包含 "direct tool"，则跳过本次大模型的思考/生成阶段，
        直接伪造一个大模型的 tool_calls 意图，强行把控制权移交给工具执行节点。
        """
        text = state["messages"][-1].content
        ## 把最后一个函数和这个函数合并了，并且把它的代码放在这里
        if "overflow" in state["messages"][-1].content:
            print("[MIDDLEWARE] before_model: jump_to='end' when contenxt window overflow")

            # 构造兜底的结束消息，并直接指定跳转到 "end" 终止 Agent 运行
            return {
                "messages":[AIMessage("上下文窗口溢出，终止")],
                "jump_to":"end",
            }

        if isinstance(text,str) and "direct tool" in text:
            print("[MIDDLEWARE] before_model: jump_to='tools'")

            # 人工构造一个大模型的消息对象（AIMessage）
            # 欺骗系统，让系统误以为这是模型自己决定要调用的工具
            fake_toolcall=AIMessage(
                content="人工构造消息",
                tool_calls=[
                    {
                        "name":"get_news",
                        "args":{},
                         "id": "call_force_weather_001",
                    }
                ]
            )

            # 返回更新后的状态：注入伪造的消息，并明确指定下一步跳转到 "tools" 节点
            return {
                "messages":[fake_toolcall],
                "jump_to":"tools",
            }
        # 如果不满足触发条件，返回 None，流程正常向下流转（继续让 LLM 思考）
        return None

    # 在模型（LLM）执行生成之后触发。允许重新跳转回 "model" 节点。
    @hook_config(can_jump_to=["model"])
    def retry_with_extra_instruction(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        """
        【业务场景：反思/重试机制】
        如果大模型已经生成了回答，但发现用户最初的请求包含 "retry model"，
        则动态追加一条系统提示词（SystemMessage），强行让模型重新生成（重试）一次。
        """
        # 倒序遍历消息历史，找到最近的一次用户输入（human 消息）
        user_text = ""
        for msg in reversed(state["messages"]):
            if getattr(msg,"type","") == "human":
                user_text = getattr(msg,"content","")
                break
        # 检查用户输入是否包含触发重试的关键字
        if isinstance(user_text,str) and "retry model" in user_text.lower():
             # 【核心防御】：防止无限循环重跳（死循环）
             # 检查消息历史中是否已经注入过这条特殊的系统提示。如果有，说明已经重试过了，不再重复干预。
             already_injected = any(
               isinstance(getattr(msg,"content",None),str)
               and "你必须以【二次回答】开头" in msg.content
               for msg in state["messages"]
             )
             if already_injected:
                 return None  # 已注入过，直接放行，结束重试流程
             print("[MIDDLEWARE] retry_with_extra_instruction: jump_to='model' with extra system instruction")
             # 返回更新后的状态：追加强力约束的系统消息，并将指针跳回 "model" 节点重新执行
             return {
                 "messages":[SystemMessage("你必须以【二次回答】开头，并且只用一句话回答。")],
                 "jump_to":"model",
             }


    # # 在模型（LLM）执行前触发。允许直接跳转到 "end" 节点（强行终止）。
    # @hook_config(can_jump_to=["end"])
    # def overflow_context_processor(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    #     """
    #     【业务场景：安全卫士/异常拦截】
    #     模拟上下文窗口溢出（Token超限）或其他严重的系统阻断情况。
    #     一旦触发，直接熔断流程，拒绝让大模型继续处理，直接报错或返回兜底文案。
    #     """
    #     # 假装溢出,模拟检查最后一条消息是否包含 overflow 标识
    #     if "overflow" in state["messages"][-1].content:
    #         print("[MIDDLEWARE] before_model: jump_to='end' when contenxt window overflow")
    #
    #         # 构造兜底的结束消息，并直接指定跳转到 "end" 终止 Agent 运行
    #         return {
    #             "messages":[AIMessage("上下文窗口溢出，终止")],
    #             "jump_to":"end",
    #         }

middleware = MyMiddleware() # 实例化中间件对象

agent = create_agent(
    model=model,
    tools=[get_news],
    # # 将定义的中间件按照顺序挂载到 Agent 中（注意：执行顺序会严格按照列表声明顺序）
    middleware=[middleware],
)

def run_once(user_input:str):
    resp = agent.invoke({
        "messages":[{"role": "user", "content": user_input}]
    })

    for msg in resp["messages"]:
        msg.pretty_print()

if __name__ == "__main__":
    # Case 1: 直接跳 tools
    # 预期表现：
    # 1. 触发 force_tool_first，打印 "[MIDDLEWARE] before_model: jump_to='tools'"
    # 2. 绕过 LLM 的首轮思考，直接调用 `get_news` 工具
    # 3. 工具返回结果后，LLM 总结工具结果并输出
    print('=' * 30, '-> Case 1 <-', '=' * 30)
    run_once("请帮我查今日新闻 direct tool")

    # Case 2: 输出后跳回 model
    # 预期表现：
    # 1. 正常进入 LLM 生成第 1 版回答
    # 2. 触发 retry_with_extra_instruction，打印 "[MIDDLEWARE] after_model: jump_to='model'..."
    # 3. 注入系统提示词后，LLM 被强行拉回并生成第 2 版回答
    # 4. 最终输出应带有“【二次回答】”前缀
    print('=' * 30, '-> Case 2 <-', '=' * 30)
    run_once("请随便介绍一下 LangChain retry model")

    # Case 3:
    # 预期表现：
    # 1. 触发 overflow_context_processor 中间件
    # 2. 直接打印终止信息并退出，LLM 根本不会接收到这个请求
    print('=' * 30, '-> Case 3 <-', '=' * 30)
    run_once("你好 overflow")
    # Case 4: 正常流程
    # 预期表现：
    # 1. 没有任何中间件被触发（不满足任何关键字）
    # 2. Agent 走正常的 OOTB（Out of the box）标准工作流：User -> Model -> Call Tool -> Model -> End
    print('=' * 30, '-> Case 4 <-', '=' * 30)
    run_once("今日新闻摘要？")

============================== -> Case 1 <- ==============================
================================ Human Message =================================

请帮我查今日新闻 direct tool
================================== Ai Message ==================================
Tool Calls:
  get_news (call_rkv0fjvu)
 Call ID: call_rkv0fjvu
  Args:
================================= Tool Message =================================
Name: get_news

美加墨世界杯今日开幕
================================== Ai Message ==================================

您好！📰 今日的头条新闻是：

**美加墨世界杯今日开幕** ⚽️✨

这是备受瞩目的国际赛事开始啦，祝您观赛愉快！
============================== -> Case 2 <- ==============================
================================ Human Message =================================

请随便介绍一下 LangChain retry model
================================== Ai Message ==================================

## LangChain Retry Model 介绍

LangChain 的重试模型（Retry Model）主要用于处理在异步任务（例如 LLM API 调用、数据获取或网络请求）过程中出现失败的情况。其设计目标是增强应用的鲁棒性，确保任务能够自动恢复并成功执行。

### 🔧 核心概念：Retr